# 06 - GARCH Volatility Modeling

Task 4: GARCH(1,1), GJR-GARCH, and EGARCH with Student-t errors (Listing 4.1), parameter interpretation (persistence, half-life, leverage), and rolling 1-day-ahead variance forecasts over the test block.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config

In [ ]:
from src.models import garch_models as gm

df = pd.read_parquet(config.PROCESSED_DATA_DIR / "prices_clean.parquet")
test = pd.read_parquet(config.PROCESSED_DATA_DIR / "test.parquet")
r = df["ret_pct"]
split = df.index.get_loc(test.index[0])  # align exactly with Task 3's test start

In [ ]:
fitted = gm.fit_garch_family(r.iloc[:split])
best_name = gm.select_best_by_bic(fitted)
print("Lowest BIC:", best_name)
for name, res in fitted.items():
    print(name, gm.summarize_params(res))

## Rolling out-of-sample variance forecasts (Listing 4.1)

In [ ]:
vf = gm.rolling_variance_forecast(r, split, gm.GARCH_SPECS[best_name], refit_every=5)
realized = gm.realized_variance_proxies(r)

ax = (vf ** 0.5).plot(figsize=(11, 4), label=f"{best_name} forecast vol")
(realized["sq_ret"].rolling(5).mean() ** 0.5).reindex(vf.index).plot(ax=ax, label="Realized (5d avg |r|)")
ax.legend(); ax.set_title("Forecast vs realized volatility - test block")
plt.show()

In [ ]:
naive_vol = gm.naive_rolling_variance_benchmark(r, split)
naive_vol.head()